# 06-C. pseudo lead time bucket model

이 노트북은 `pre_fault` 구간 내부에서 `0-6h / 6-24h / 1-3d / 3-7d` pseudo lead time bucket을 예측하는 LightGBM 분류 baseline을 학습한다.


In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 240)


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for path in [current, *current.parents]:
        if (path / 'pyproject.toml').exists():
            return path
    raise FileNotFoundError('pyproject.toml not found. Run from the project tree.')


PROJECT_ROOT = find_project_root(Path.cwd())
FEATURE_DIR = PROJECT_ROOT / 'data' / 'processed' / 'ml_features'
RISK_DIR = PROJECT_ROOT / 'data' / 'processed' / 'ml_risk'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed' / 'ml_leadtime'
MODEL_DIR = OUTPUT_DIR / 'models'

TRAINABLE_WINDOWS_PATH = FEATURE_DIR / 'trainable_windows.csv'
FEATURE_COLUMNS_PATH = FEATURE_DIR / 'feature_columns.csv'
RISK_SCORES_PATH = RISK_DIR / 'lgbm_risk_scores.csv'
RISK_MODEL_METADATA_PATH = RISK_DIR / 'models' / 'risk_model_metadata.json'

LEADTIME_SCORES_PATH = OUTPUT_DIR / 'leadtime_bucket_scores.csv'
LEADTIME_METRICS_PATH = OUTPUT_DIR / 'leadtime_bucket_metrics.csv'
LEADTIME_CONFUSION_PATH = OUTPUT_DIR / 'leadtime_bucket_confusion_matrix.csv'
LEADTIME_MODEL_PATH = MODEL_DIR / 'lightgbm_leadtime_bucket_model.joblib'
LEADTIME_METADATA_PATH = MODEL_DIR / 'leadtime_bucket_model_metadata.json'

KEY_COLUMNS = ['manufacturer', 'substation_id', 'window_start', 'window_end']
LEADTIME_LABELS = ['0-24h', '1-3d', '3-7d']
LEADTIME_LABEL_TO_INDEX = {label: index for index, label in enumerate(LEADTIME_LABELS)}
LEADTIME_BUCKET_MAPPING = {'0-6h': '0-24h', '6-24h': '0-24h', '1-3d': '1-3d', '3-7d': '3-7d'}
MODEL_VERSION = 'lgbm_leadtime_bucket_v2_3bucket'
PRIMARY_SPLIT_COLUMN = 'split_event_regime_based'
SPLIT_VALUES = ['train', 'validation', 'holdout']
RANDOM_STATE = 42

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f'trainable windows: {TRAINABLE_WINDOWS_PATH}')
print(f'risk scores: {RISK_SCORES_PATH}')
print(f'output dir: {OUTPUT_DIR}')


trainable windows: C:\Project3\HeatGrid_Agent\data\processed\ml_features\trainable_windows.csv
risk scores: C:\Project3\HeatGrid_Agent\data\processed\ml_risk\lgbm_risk_scores.csv
output dir: C:\Project3\HeatGrid_Agent\data\processed\ml_leadtime


In [2]:
trainable_windows = pd.read_csv(TRAINABLE_WINDOWS_PATH)
feature_columns = pd.read_csv(FEATURE_COLUMNS_PATH)
risk_scores = pd.read_csv(RISK_SCORES_PATH)
risk_model_metadata = json.loads(RISK_MODEL_METADATA_PATH.read_text(encoding='utf-8'))

selected_feature_columns = (
    feature_columns.loc[feature_columns['selected_for_baseline'] == True, 'column_name']
    .dropna()
    .tolist()
)
cumulative_absolute_feature_columns = {
    f'p_net_meter_{meter}__{stat}'
    for meter in ['energy', 'volume']
    for stat in ['first', 'last', 'min', 'max', 'mean']
}
selected_feature_columns = [
    column for column in selected_feature_columns
    if column not in cumulative_absolute_feature_columns
]

risk_context_columns = [
    column for column in [
        'anomaly_score',
        'risk_probability',
        'risk_score',
        'disturbance_count',
        'maintenance_related',
        'days_since_last_fault_event',
        'days_since_last_task_event',
        'days_since_last_any_event',
        'split_event_based',
        'split_event_regime_based',
        'split_regime_based',
        'split_time_based',
        'split_substation_based',
        'lead_time_bucket',
        'estimated_lead_time_hours',
        'risk_level',
        'label',
        'fault_event_id',
        'fault_label',
        'configuration_type',
    ]
    if column in risk_scores.columns
]
trainable_feature_columns = [
    column for column in selected_feature_columns
    if column in trainable_windows.columns and column not in risk_context_columns
]

modeling_df = risk_scores[KEY_COLUMNS + risk_context_columns].merge(
    trainable_windows[KEY_COLUMNS + [column for column in trainable_feature_columns if column not in KEY_COLUMNS]],
    on=KEY_COLUMNS,
    how='left',
    validate='one_to_one',
)

if 'manufacturer' in modeling_df.columns:
    modeling_df['manufacturer_code'] = modeling_df['manufacturer'].astype('category').cat.codes.astype('int16')
if 'configuration_type' in modeling_df.columns:
    modeling_df['configuration_code'] = (
        modeling_df['configuration_type']
        .fillna('missing')
        .astype('category')
        .cat.codes
        .astype('int16')
    )

for column in ['maintenance_related']:
    if column in modeling_df.columns:
        modeling_df[column] = modeling_df[column].fillna(False).astype('int8')

for column in ['disturbance_count']:
    if column in modeling_df.columns:
        modeling_df[column] = modeling_df[column].fillna(0)

modeling_df['lead_time_bucket_3'] = modeling_df['lead_time_bucket'].map(LEADTIME_BUCKET_MAPPING)
pre_fault_df = modeling_df.loc[
    modeling_df['label'].eq('pre_fault')
    & modeling_df['lead_time_bucket_3'].isin(LEADTIME_LABELS)
].copy()
pre_fault_df['lead_time_bucket'] = pre_fault_df['lead_time_bucket_3']
pre_fault_df['lead_time_target'] = pre_fault_df['lead_time_bucket'].map(LEADTIME_LABEL_TO_INDEX).astype(int)

model_feature_columns = []
for column in [
    *trainable_feature_columns,
    'anomaly_score',
    'risk_probability',
    'risk_score',
    'disturbance_count',
    'maintenance_related',
    'days_since_last_fault_event',
    'days_since_last_task_event',
    'days_since_last_any_event',
    'manufacturer_code',
    'configuration_code',
]:
    if column in pre_fault_df.columns and column not in model_feature_columns:
        model_feature_columns.append(column)

X_all = pre_fault_df[model_feature_columns].copy()
for column in X_all.columns:
    if X_all[column].dtype == 'bool':
        X_all[column] = X_all[column].astype('int8')
    elif X_all[column].dtype == 'object':
        X_all[column] = pd.to_numeric(X_all[column], errors='coerce')

if X_all.isna().any().any():
    missing_summary = X_all.isna().sum()
    missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)
    raise ValueError('Leadtime model input features contain missing values:\n' + str(missing_summary.head(20)))

y_all = pre_fault_df['lead_time_target']
train_mask = pre_fault_df[PRIMARY_SPLIT_COLUMN].eq('train')
validation_mask = pre_fault_df[PRIMARY_SPLIT_COLUMN].eq('validation')
holdout_mask = pre_fault_df[PRIMARY_SPLIT_COLUMN].eq('holdout')

print(f'pre_fault rows: {len(pre_fault_df)}')
print(pre_fault_df.groupby(PRIMARY_SPLIT_COLUMN)['lead_time_bucket'].value_counts().to_string())
print(f'feature count: {len(model_feature_columns)}')


pre_fault rows: 636
split_event_regime_based  lead_time_bucket
holdout                   1-3d                 48
                          0-24h                35
                          3-7d                  3
train                     1-3d                234
                          0-24h               137
                          3-7d                 36
validation                1-3d                 74
                          0-24h                39
                          3-7d                 30
feature count: 193


C:\Users\Admin\AppData\Local\Temp\ipykernel_18456\1609510502.py:59: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  modeling_df['manufacturer_code'] = modeling_df['manufacturer'].astype('category').cat.codes.astype('int16')
C:\Users\Admin\AppData\Local\Temp\ipykernel_18456\1609510502.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  modeling_df['configuration_code'] = (
C:\Users\Admin\AppData\Local\Temp\ipykernel_18456\1609510502.py:77: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `

In [3]:
leadtime_model = LGBMClassifier(
    objective='multiclass',
    num_class=len(LEADTIME_LABELS),
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=15,
    min_child_samples=20,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.1,
    reg_lambda=1.0,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1,
)

leadtime_model.fit(
    X_all.loc[train_mask],
    y_all.loc[train_mask],
    eval_set=[(X_all.loc[validation_mask], y_all.loc[validation_mask])],
    eval_metric='multi_logloss',
)

probabilities = leadtime_model.predict_proba(X_all)
predicted_index = probabilities.argmax(axis=1)
predicted_bucket = [LEADTIME_LABELS[index] for index in predicted_index]
predicted_confidence = probabilities.max(axis=1)

pre_fault_df['predicted_lead_time_bucket'] = predicted_bucket
pre_fault_df['predicted_lead_time_confidence'] = predicted_confidence
pre_fault_df['predicted_lead_time_index'] = predicted_index
pre_fault_df['lead_time_bucket_distance'] = (
    pre_fault_df['predicted_lead_time_index'] - pre_fault_df['lead_time_target']
).abs()

for index, label in enumerate(LEADTIME_LABELS):
    pre_fault_df[f'leadtime_prob_{label}'] = probabilities[:, index]


def top2_accuracy(y_true: np.ndarray, proba: np.ndarray) -> float:
    top2 = np.argsort(proba, axis=1)[:, -2:]
    hits = [(target in top2_row) for target, top2_row in zip(y_true, top2)]
    return float(np.mean(hits))


metric_rows = []
confusion_rows = []
for split_value in SPLIT_VALUES:
    frame = pre_fault_df.loc[pre_fault_df[PRIMARY_SPLIT_COLUMN] == split_value].copy()
    if frame.empty:
        continue
    y_true = frame['lead_time_target'].to_numpy()
    y_pred = frame['predicted_lead_time_index'].to_numpy()
    proba = frame[[f'leadtime_prob_{label}' for label in LEADTIME_LABELS]].to_numpy()
    precision, recall, f1_per_class, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=list(range(len(LEADTIME_LABELS))),
        zero_division=0,
    )
    metric_rows.append(
        {
            'split': split_value,
            'row_count': len(frame),
            'accuracy': float(accuracy_score(y_true, y_pred)),
            'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
            'weighted_f1': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
            'top2_accuracy': top2_accuracy(y_true, proba),
            'bucket_distance_mae': float(frame['lead_time_bucket_distance'].mean()),
        }
    )
    matrix = confusion_matrix(y_true, y_pred, labels=list(range(len(LEADTIME_LABELS))))
    for actual_index, actual_label in enumerate(LEADTIME_LABELS):
        for predicted_index_value, predicted_label_value in enumerate(LEADTIME_LABELS):
            confusion_rows.append(
                {
                    'split': split_value,
                    'actual_bucket': actual_label,
                    'predicted_bucket': predicted_label_value,
                    'count': int(matrix[actual_index, predicted_index_value]),
                }
            )

metrics_df = pd.DataFrame(metric_rows)
confusion_df = pd.DataFrame(confusion_rows)

leadtime_scores_df = pre_fault_df[[
    'manufacturer',
    'substation_id',
    'window_start',
    'window_end',
    'fault_event_id',
    'fault_label',
    'estimated_lead_time_hours',
    'lead_time_bucket',
    'predicted_lead_time_bucket',
    'predicted_lead_time_confidence',
    'lead_time_bucket_distance',
    'anomaly_score',
    'risk_probability',
    'risk_level',
    'days_since_last_fault_event',
    'days_since_last_task_event',
    'days_since_last_any_event',
    'split_event_based',
    'split_event_regime_based',
    'split_regime_based',
    'split_time_based',
    'split_substation_based',
    *[f'leadtime_prob_{label}' for label in LEADTIME_LABELS],
]].copy()
leadtime_scores_df['model_version'] = MODEL_VERSION

model_metadata = {
    'model_version': MODEL_VERSION,
    'model_type': 'LightGBM LGBMClassifier multiclass',
    'target_definition': 'pseudo lead time bucket within pre_fault rows only',
    'leadtime_labels': LEADTIME_LABELS,
    'primary_split_column': PRIMARY_SPLIT_COLUMN,
    'input_trainable_windows_path': str(TRAINABLE_WINDOWS_PATH),
    'input_risk_scores_path': str(RISK_SCORES_PATH),
    'input_risk_model_metadata_path': str(RISK_MODEL_METADATA_PATH),
    'feature_count': len(model_feature_columns),
    'model_feature_columns': model_feature_columns,
    'risk_model_version': risk_model_metadata.get('model_version'),
    'metrics': metrics_df.to_dict(orient='records'),
}

leadtime_scores_df.to_csv(LEADTIME_SCORES_PATH, index=False)
metrics_df.to_csv(LEADTIME_METRICS_PATH, index=False)
confusion_df.to_csv(LEADTIME_CONFUSION_PATH, index=False)
joblib.dump(leadtime_model, LEADTIME_MODEL_PATH)
LEADTIME_METADATA_PATH.write_text(json.dumps(model_metadata, ensure_ascii=False, indent=2), encoding='utf-8')

display(metrics_df)
display(confusion_df.head(16))
print(LEADTIME_SCORES_PATH)
print(LEADTIME_METRICS_PATH)
print(LEADTIME_CONFUSION_PATH)
print(LEADTIME_MODEL_PATH)
print(LEADTIME_METADATA_PATH)


,split,row_count,accuracy,macro_f1,weighted_f1,top2_accuracy,bucket_distance_mae
0,train,407,1.000000,1.000000,1.000000,1.000000,0.000000
1,validation,143,0.433566,0.319608,0.398272,0.790210,0.713287
2,holdout,86,0.651163,0.432900,0.638478,0.965116,0.383721


,split,actual_bucket,predicted_bucket,count
0,train,0-24h,0-24h,137
1,train,0-24h,1-3d,0
2,train,0-24h,3-7d,0
3,train,1-3d,0-24h,0
4,train,1-3d,1-3d,234
5,train,1-3d,3-7d,0
6,train,3-7d,0-24h,0
7,train,3-7d,1-3d,0
8,train,3-7d,3-7d,36
9,validation,0-24h,0-24h,24


C:\Project3\HeatGrid_Agent\data\processed\ml_leadtime\leadtime_bucket_scores.csv
C:\Project3\HeatGrid_Agent\data\processed\ml_leadtime\leadtime_bucket_metrics.csv
C:\Project3\HeatGrid_Agent\data\processed\ml_leadtime\leadtime_bucket_confusion_matrix.csv
C:\Project3\HeatGrid_Agent\data\processed\ml_leadtime\models\lightgbm_leadtime_bucket_model.joblib
C:\Project3\HeatGrid_Agent\data\processed\ml_leadtime\models\leadtime_bucket_model_metadata.json


## 2026-06-25 leadtime promotion note

- promoted candidate: `3-bucket + timeflow_lag_delta_roll3`
- holdout improved slightly versus baseline
- baseline holdout macro F1: `0.4329`
- promoted holdout macro F1: `0.4405`
- promotion review doc: `PREPROCESSING/docs/06_leadtime_promotion_decision.md`
